# Notebook 57: Bitcoin Valuation Models (Bitcoin Lab API)

**Core Principle:** Seek CONFLUENCE between multiple models.

**Data Source:** Bitcoin Lab API (https://api.researchbitcoin.net)

**Models Implemented:**
- Cost Basis: Realized Price, True Market Mean, STH Cost Basis, Vaulted Price
- Ratios: MVRV, AVIV
- Production: Thermocap, Difficulty
- Technical: 200-day MA, 200-week MA, Power Law

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

DATA_DIR = Path("../data")
DAILY_DIR = DATA_DIR / "daily"

## 1. Load Bitcoin Lab Data

In [ ]:
def load_metric(name):
    """Load a metric from local parquet files (downloaded via Bitcoin Lab API)"""
    path = DAILY_DIR / f"{name}.parquet"
    if path.exists():
        df = pd.read_parquet(path).set_index("time")
        # Convert to timezone-naive for consistency
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        return df["value"] if "value" in df.columns else df.iloc[:, 0]
    else:
        print(f"  ⚠️ Missing: {name}")
        return None

# Check available metrics
available = list(DAILY_DIR.glob("*.parquet"))
print(f"Available metrics: {len(available)}")
for f in sorted(available)[:10]:
    print(f"  {f.stem}")
print("  ...")

In [ ]:
# Load core metrics
print("\nLoading valuation metrics...")

metrics = {
    # Price
    "price": load_metric("price"),
    
    # Cost Basis Models (from Bitcoin Lab API)
    "realized_price": load_metric("realized_price"),
    "realized_price_sth": load_metric("realized_price_sth"),
    "realized_price_lth": load_metric("realized_price_lth"),
    "true_market_mean": load_metric("true_market_mean_price"),
    "vaulted_price": load_metric("vaulted_price"),
    
    # Ratios
    "mvrv": load_metric("mvrv"),
    "mvrv_z": load_metric("mvrv_z"),
    "aviv": load_metric("aviv"),
    
    # Production Cost
    "thermo_cap": load_metric("thermo_cap"),
    "difficulty": load_metric("difficulty"),
    "hashrate": load_metric("hashrate"),
    
    # Supply
    "supply_total": load_metric("supply_total"),
    "market_cap": load_metric("market_cap"),
    "realized_cap": load_metric("realized_cap"),
}

# Report what we have
loaded = {k: v for k, v in metrics.items() if v is not None}
missing = {k: v for k, v in metrics.items() if v is None}
print(f"\n✅ Loaded: {len(loaded)} metrics")
print(f"❌ Missing: {len(missing)} metrics")
if missing:
    print(f"   Run: python run.py download --metrics {','.join(missing.keys())}")

In [ ]:
# Build combined dataframe
df = pd.DataFrame(metrics).dropna(how='all')

# Ensure index is timezone-naive
if df.index.tz is not None:
    df.index = df.index.tz_localize(None)

# Calculate technical indicators
if "price" in df.columns:
    df["ma_200d"] = df["price"].rolling(200).mean()
    df["ma_200w"] = df["price"].rolling(200*7).mean()  # ~1400 days
    
    # Power law trend (simplified)
    genesis = pd.Timestamp("2009-01-03")
    df["days_since_genesis"] = (df.index - genesis).days
    log_days = np.log10(df["days_since_genesis"].clip(lower=1))
    log_price = np.log10(df["price"].clip(lower=0.01))
    
    # Fit on data after 2013 for stability
    mask = (df.index >= "2013-01-01") & log_days.notna() & log_price.notna()
    if mask.sum() > 100:
        coeffs = np.polyfit(log_days[mask], log_price[mask], 1)
        df["power_law_trend"] = 10 ** (coeffs[0] * log_days + coeffs[1])
        print(f"Power law: price = 10^({coeffs[1]:.2f} + {coeffs[0]:.4f} * log10(days))")

print(f"\nCombined data: {len(df)} days")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

## 2. Current Valuation Status

In [ ]:
# Get latest values
current = df.iloc[-1]
price = current.get("price", np.nan)

print("="*70)
print("CURRENT VALUATION STATUS")
print("="*70)
print(f"\nDate: {df.index[-1].date()}")
print(f"Price: ${price:,.0f}")

# Cost basis levels
print("\n--- Cost Basis Models ---")
cost_basis_metrics = [
    ("Realized Price (Bear Floor)", "realized_price"),
    ("True Market Mean (Fair Value)", "true_market_mean"),
    ("STH Realized Price (Sentiment)", "realized_price_sth"),
    ("LTH Realized Price", "realized_price_lth"),
    ("Vaulted Price (Bull Ceiling)", "vaulted_price"),
]

for name, key in cost_basis_metrics:
    val = current.get(key)
    if pd.notna(val) and val > 0:
        pct = (price / val - 1) * 100
        print(f"  {name:<35}: ${val:>10,.0f}  ({pct:>+6.1f}%)")
    else:
        print(f"  {name:<35}: [Not available]")

# Technical levels
print("\n--- Technical Models ---")
tech_metrics = [
    ("200-Day MA", "ma_200d"),
    ("200-Week MA", "ma_200w"),
    ("Power Law Trend", "power_law_trend"),
]

for name, key in tech_metrics:
    val = current.get(key)
    if pd.notna(val) and val > 0:
        pct = (price / val - 1) * 100
        print(f"  {name:<35}: ${val:>10,.0f}  ({pct:>+6.1f}%)")

# Ratios
print("\n--- Valuation Ratios ---")
ratio_metrics = [
    ("MVRV", "mvrv", 1.0),
    ("MVRV Z-Score", "mvrv_z", 0.0),
    ("AVIV", "aviv", 1.0),
]

for name, key, neutral in ratio_metrics:
    val = current.get(key)
    if pd.notna(val):
        status = "🟢 Undervalued" if val < neutral else "🔴 Overvalued" if val > neutral * 1.5 else "🟡 Fair"
        print(f"  {name:<20}: {val:>8.2f}  (neutral={neutral}) {status}")

## 3. Confluence Analysis

In [ ]:
print("\n" + "="*70)
print("CONFLUENCE CHECK")
print("="*70)

# Define confluence signals
signals = []

# Cost basis signals
if pd.notna(current.get("realized_price")):
    if price < current["realized_price"]:
        signals.append(("✅ STRONG BUY", "Price below Realized Price"))
    elif price < current["realized_price"] * 1.2:
        signals.append(("✅ BUY", "Price near Realized Price (<20% above)"))

if pd.notna(current.get("true_market_mean")):
    if price < current["true_market_mean"]:
        signals.append(("✅ BUY", "Price below True Market Mean"))
    elif price < current["true_market_mean"] * 1.1:
        signals.append(("🟡 NEUTRAL", "Price at Fair Value (TMM)"))

if pd.notna(current.get("realized_price_sth")):
    if price < current["realized_price_sth"]:
        signals.append(("✅ BUY", "Price below STH Cost Basis (capitulation)"))
    elif price > current["realized_price_sth"] * 1.3:
        signals.append(("🔴 CAUTION", "Price >30% above STH Cost Basis"))

if pd.notna(current.get("vaulted_price")):
    if price > current["vaulted_price"]:
        signals.append(("🔴 SELL", "Price above Vaulted Price (distribution zone)"))

# Technical signals
if pd.notna(current.get("ma_200d")):
    if price < current["ma_200d"]:
        signals.append(("✅ BUY", "Price below 200-day MA"))

if pd.notna(current.get("power_law_trend")):
    if price < current["power_law_trend"] * 0.5:
        signals.append(("✅ STRONG BUY", "Price >50% below Power Law"))
    elif price < current["power_law_trend"]:
        signals.append(("✅ BUY", "Price below Power Law trend"))

# Ratio signals
if pd.notna(current.get("mvrv")):
    if current["mvrv"] < 1:
        signals.append(("✅ STRONG BUY", "MVRV < 1 (market below cost basis)"))
    elif current["mvrv"] > 3:
        signals.append(("🔴 SELL", "MVRV > 3 (historically overvalued)"))

if pd.notna(current.get("aviv")):
    if current["aviv"] < 1:
        signals.append(("✅ BUY", "AVIV < 1 (active investors underwater)"))

# Count signals
buy_signals = len([s for s in signals if "BUY" in s[0]])
sell_signals = len([s for s in signals if "SELL" in s[0] or "CAUTION" in s[0]])
neutral_signals = len([s for s in signals if "NEUTRAL" in s[0]])

print(f"\nSignals Found: {len(signals)}")
print(f"  🟢 Buy:     {buy_signals}")
print(f"  🔴 Sell:    {sell_signals}")
print(f"  🟡 Neutral: {neutral_signals}")

print("\nSignal Details:")
for status, reason in signals:
    print(f"  {status}: {reason}")

# Confluence verdict
print("\n" + "="*70)
if buy_signals >= 4:
    print("CONFLUENCE VERDICT: 🟢 STRONG BUY (4+ confirmations)")
elif buy_signals >= 2 and sell_signals == 0:
    print("CONFLUENCE VERDICT: 🟢 BUY (2-3 confirmations)")
elif sell_signals >= 3:
    print("CONFLUENCE VERDICT: 🔴 SELL (multiple warnings)")
elif sell_signals >= 1 and buy_signals <= 1:
    print("CONFLUENCE VERDICT: 🟡 CAUTION (mixed signals)")
else:
    print("CONFLUENCE VERDICT: 🟡 NEUTRAL (no strong confluence)")
print("="*70)

## 4. Valuation Zone Chart

In [ ]:
# Only plot if we have the key metrics
required = ["price", "realized_price", "true_market_mean", "realized_price_sth"]
if all(k in df.columns and df[k].notna().any() for k in required):
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Filter to recent years
    plot_df = df[df.index >= "2020-01-01"].copy()
    
    # Plot price
    ax.semilogy(plot_df.index, plot_df["price"], 'k-', linewidth=2, label="Price")
    
    # Plot cost basis levels
    if "realized_price" in plot_df.columns:
        ax.semilogy(plot_df.index, plot_df["realized_price"], 'r--', 
                    linewidth=1.5, label="Realized Price (Bear Floor)")
    
    if "true_market_mean" in plot_df.columns:
        ax.semilogy(plot_df.index, plot_df["true_market_mean"], 'g-', 
                    linewidth=2, label="True Market Mean (Fair Value)")
    
    if "realized_price_sth" in plot_df.columns:
        ax.semilogy(plot_df.index, plot_df["realized_price_sth"], 'b--', 
                    linewidth=1.5, label="STH Realized Price")
    
    if "vaulted_price" in plot_df.columns:
        ax.semilogy(plot_df.index, plot_df["vaulted_price"], 'm--', 
                    linewidth=1.5, label="Vaulted Price (Bull Ceiling)")
    
    if "ma_200d" in plot_df.columns:
        ax.semilogy(plot_df.index, plot_df["ma_200d"], 'orange', 
                    linewidth=1, alpha=0.7, label="200-Day MA")
    
    ax.set_title("Bitcoin Valuation Models (Seeking Confluence)")
    ax.set_ylabel("Price (USD, log scale)")
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Missing required metrics for chart. Run the downloader first.")
    print(f"   Required: {required}")

## 5. Integration with Trading Strategies

In [ ]:
print("\n" + "="*70)
print("STRATEGY INTEGRATION")
print("="*70)

# Position sizing based on valuation
def get_position_multiplier(price, current):
    """Returns position size multiplier based on valuation zone"""
    rp = current.get("realized_price", 0)
    tmm = current.get("true_market_mean", 0)
    sth = current.get("realized_price_sth", 0)
    vp = current.get("vaulted_price", float('inf'))
    
    if price < rp and rp > 0:
        return 2.0, "EXTREME BEAR - Max position"
    elif price < tmm and tmm > 0:
        return 1.5, "UNDERVALUED - Increased position"
    elif price < sth and sth > 0:
        return 1.0, "FAIR VALUE - Standard position"
    elif price < vp:
        return 0.5, "OVERVALUED - Reduced position"
    else:
        return 0.25, "EXTREME BULL - Minimal position"

mult, zone = get_position_multiplier(price, current)

print(f"\nCurrent Price: ${price:,.0f}")
print(f"Valuation Zone: {zone}")
print(f"Position Multiplier: {mult}x")

print("\n--- Enhanced Entry Rules ---")
print("""
STRAT-002/004 Entry Signal fires PLUS:

  IF price ≤ Realized Price:
     → 2x position (rare, max conviction)
  
  IF price ≤ True Market Mean:
     → 1.5x position (strong conviction)
  
  IF price ≤ STH Realized Price:
     → 1x position (standard)
  
  IF price > STH Realized Price:
     → 0.5x position (reduced)
  
  IF price > Vaulted Price:
     → Skip entry or 0.25x only
""")

## 6. Save Current Valuation State

In [ ]:
# Save current state for other notebooks
valuation_state = {
    "timestamp": datetime.now().isoformat(),
    "date": str(df.index[-1].date()),
    "price": float(price) if pd.notna(price) else None,
    "models": {},
    "confluence": {
        "buy_signals": buy_signals,
        "sell_signals": sell_signals,
        "total_signals": len(signals),
    },
    "position_multiplier": mult,
    "zone": zone,
}

# Add all available model values
for name in ["realized_price", "true_market_mean", "realized_price_sth", 
             "realized_price_lth", "vaulted_price", "ma_200d", "ma_200w",
             "power_law_trend", "mvrv", "aviv"]:
    val = current.get(name)
    if pd.notna(val):
        valuation_state["models"][name] = float(val)

# Save
output_path = DATA_DIR / "valuation_state.json"
with open(output_path, "w") as f:
    json.dump(valuation_state, f, indent=2)

print(f"\n✅ Saved valuation state to {output_path}")
print(f"   Use in other notebooks: json.load(open('data/valuation_state.json'))")

## 7. Summary

In [ ]:
print("\n" + "="*70)
print("VALUATION FRAMEWORK SUMMARY")
print("="*70)

print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    CONFLUENCE PRINCIPLE                              ║
║                                                                      ║
║  "Whenever we can identify confluence between several models,        ║
║   it can help to add confidence about the trends playing out."      ║
║                                              - James Check           ║
╚══════════════════════════════════════════════════════════════════════╝

VALUATION HIERARCHY (from Bitcoin Lab API):

  🔴 EXTREME BEAR  │ Below Realized Price      │ 2x position
  ─────────────────┼───────────────────────────┼────────────
  🟠 BEAR          │ RP → Thermocap multiple   │ 1.5x position
  ─────────────────┼───────────────────────────┼────────────
  🟡 UNDERVALUED   │ Below True Market Mean    │ 1.5x position
  ─────────────────┼───────────────────────────┼────────────
  🟢 FAIR VALUE    │ TMM → STH Realized Price  │ 1x position
  ─────────────────┼───────────────────────────┼────────────
  🟡 OVERVALUED    │ Above STH RP              │ 0.5x position
  ─────────────────┼───────────────────────────┼────────────
  🔵 BULL          │ Approaching Vaulted       │ 0.25x / exit
  ─────────────────┼───────────────────────────┼────────────
  🟣 EXTREME BULL  │ Above Vaulted Price       │ Distribution

CONFLUENCE CHECKLIST:
  □ Cost Basis Models (RP, TMM, STH RP)
  □ Valuation Ratios (MVRV, AVIV)
  □ Technical Levels (200D MA, Power Law)
  □ On-Chain Signals (SOPR, RL z-score)
  
  4+ confirmations = High confidence trade
  2-3 confirmations = Moderate confidence
  1 confirmation = Monitor only
""")